In [2]:
import kes

Importing...
... Done


In [3]:
import importlib
importlib.reload(kes)
models = kes.ComModels()
cache = kes.ComCache()

Importing...
... Done
VAE...
CLIP...
VACE UNet...
VACE CausVid...
...Done


In [ ]:
import random
import torch
import os
import time
import comfy.utils
import importlib
import numpy as np
importlib.reload(kes)

pos_text = """
ultra-sharp detailed 8K realistic video of a topless woman relaxing on sunlit seaside  
rocks, lying on a patterned towel near the water, smiling naturally, wet dark  
hair, turquoise and white bikini bottoms, white towel beside her, 
golden sunlight shining on her skin and rocks, 
vivid green pine trees and blue sea in the background, calm  
Mediterranean light. She slowly rolls onto her back to tan, exhibit her huge boobs,
moving naturally  and comfortably, relaxing with eyes half-closed, she has beautiful breasts,
with large dark areolas, erect nipples,  

elle se retourne et s'allonge sur le dos, ses seins énormes au soleil.

waves glittering softly behind, cinematic lighting, HDR, smooth natural  
motion, photorealistic detail, perfect sharpness, masterpiece, best quality.
"""

neg_text = """
low quality, blurry, flicker, low contrast, overexposed, underexposed, noise,  
distorted anatomy, stiff motion, unrealistic skin, painting, cartoon, CGI,  
3D render, extra limbs, watermark, text, logo, color banding, compression  
artifacts, unnatural camera shake
"""

for i in range(1, 81000000, 1): #linspace(1.0, 6.0, 5):
    length = 81
    cfg = 1.0 #int(random.uniform(1.0, 2.0) * 100) / 100
    steps = random.randint(4, 8)
    shift = random.randint(2, 6)
    width, height = 720, 480
    img = kes.load_image("/workspace/out5/DSC07410.JPG")
    img = comfy.utils.common_upscale(img.movedim(-1, 1), width, height, "bilinear", "center").movedim(1, -1)[:,:,:,:3]
    control_video = torch.ones((length, height, width, 3)) #, dtype=vid.dtype, device=vid.device)
    control_masks = torch.ones((length, height, width)) #, dtype=vid.dtype, device=vid.device)
    control_video[0] = img[0]
    control_masks[0] = 0

    # ref_img = kes.load_image("/workspace/out5/Screenshot 2025-11-12 at 02.15.51.png")
    # ref_img = comfy.utils.common_upscale(ref_img.movedim(-1, 1), width, height, "bilinear", "center").movedim(1, -1)[:,:,:,:3]
    ref_img = None
    out_images = kes.com_vace(
            seed=12346+i, steps=steps, shift=shift,
            width=width, height=height, length=length,
            cfg=cfg, sampler="unipc", scheduler="simple",
            control_video = control_video,
            control_masks = None,
            reference_image = ref_img,
            loras=[],
            pos_text = pos_text,
            neg_text = neg_text,
            models=models, cache=cache, causvid=True
        )
    ts = int(time.time())
    fname = f"/workspace/out6/vace-nude-ts={ts}-cfg={cfg}-stp={steps}-sft={shift}"
    # kes.save(out_images[:1], f"/workspace/vace-in-first")
    # kes.save(out_images[-1:], f"/workspace/vace-in-last")
    kes.save(out_images, fname, fps=16)

Importing...
... Done
Loras
  cache hit
Set shift
Encoding pos
  cache hit
Encoding neg
  cache hit
Encoding latent
Sampling


  0%|          | 0/5 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 554.00 MiB. GPU 0 has a total capacity of 31.37 GiB of which 513.50 MiB is free. Process 115 has 496.00 MiB memory in use. Including non-PyTorch memory, this process has 30.37 GiB memory in use. Of the allocated memory 28.95 GiB is allocated by PyTorch, and 840.05 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [5]:
480*16//9//16*16

848